# Sentiment Analysis: NLP Classification Case Study

Classify text as **positive** or **negative** sentiment using classic machine learning pipelines.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import re
import string
import warnings
warnings.filterwarnings('ignore')

from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.model_selection import cross_val_score, train_test_split, StratifiedKFold
from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC, LinearSVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import confusion_matrix, classification_report, roc_curve, auc
from sklearn.pipeline import Pipeline

from nltk.corpus import stopwords
from nltk.stem import PorterStemmer

sns.set_style('whitegrid')
%matplotlib inline

In [ ]:
import nltk
nltk.download('stopwords')
nltk.download('wordnet')
nltk.download('punkt')
print('NLTK downloads complete.')

---
## 1. Problem Statement

Given a piece of text, predict whether it expresses **positive** or **negative** sentiment.
We will demonstrate the full NLP pipeline: raw text -> preprocessing -> feature extraction -> model training -> evaluation.

---
## 2. Create Synthetic Text Data

Hardcoded sample sentences (no downloads).

In [ ]:
positive_sentences = [
    'This movie was absolutely fantastic and inspiring',
    'I loved every minute of this wonderful experience',
    'The product exceeded my expectations, truly amazing',
    'Great service and very friendly staff',
    'An outstanding performance by the entire team',
    'The food was delicious and the ambiance was perfect',
    'I am so happy with the results, highly recommend',
    'Beautiful scenery and breathtaking views',
    'The customer support was helpful and quick to respond',
    'A brilliant masterpiece that everyone should see',
    'Wonderful experience, will definitely come back',
    'The quality is superb and the price is fair',
    'I feel great after using this product every day',
    'Excellent course with clear explanations',
    'The team did a phenomenal job on this project',
    'Such a delightful and charming little cafe',
    'Perfect vacation, everything was well organized',
    'I am grateful for this amazing opportunity',
    'The design is elegant and user friendly',
    'Best purchase I have made all year'
]

negative_sentences = [
    'This was a terrible waste of time and money',
    'I hated every second of this boring film',
    'The product broke after one use, very disappointed',
    'Rude staff and horrible customer service',
    'A poorly written script with awful acting',
    'The food was cold and the service was slow',
    'I regret buying this, it was a complete failure',
    'Dirty rooms and noisy environment, avoid at all costs',
    'The worst experience I have ever had',
    'Terrible quality, does not work as advertised',
    'This app crashes constantly, very frustrating',
    'The delivery was late and the item was damaged',
    'I feel awful after taking this medication',
    'Confusing instructions and poor support',
    'A disappointing outcome despite the initial promise',
    'The interface is clunky and unintuitive',
    'Overpriced and not worth the hype',
    'I will never use this service again',
    'The build quality is shoddy and cheap',
    'Nothing but problems since day one'
]

sentences = positive_sentences + negative_sentences
labels = ['positive'] * len(positive_sentences) + ['negative'] * len(negative_sentences)

df = pd.DataFrame({'text': sentences, 'label': labels})
print(f'Dataset size: {len(df)} samples')
print(df['label'].value_counts())

In [ ]:
df.head(8)

---
## 3. Text Preprocessing

Steps: lowercase, remove punctuation & digits, remove stopwords, apply stemming.

In [ ]:
stop_words = set(stopwords.words('english'))
stemmer = PorterStemmer()

def preprocess(text):
    text = text.lower()
    text = re.sub(f'[{re.escape(string.punctuation)}]', '', text)
    text = re.sub(r'\d+', '', text)
    tokens = text.split()
    tokens = [t for t in tokens if t not in stop_words]
    tokens = [stemmer.stem(t) for t in tokens]
    return ' '.join(tokens)

df['clean_text'] = df['text'].apply(preprocess)
df[['text', 'clean_text']].head(5)

---
## 4. Feature Extraction

Comparing Bag of Words (CountVectorizer) and TF-IDF with unigrams and bigrams.

In [ ]:
cv = CountVectorizer(ngram_range=(1, 2))
tfidf = TfidfVectorizer(ngram_range=(1, 2), max_features=500)

X_cv = cv.fit_transform(df['clean_text'])
X_tfidf = tfidf.fit_transform(df['clean_text'])
y = np.array([1 if l == 'positive' else 0 for l in df['label']])

print(f'BoW feature matrix shape: {X_cv.shape}')
print(f'TF-IDF feature matrix shape: {X_tfidf.shape}')
print(f'Vocabulary size: {len(cv.get_feature_names_out())}')

---
## 5. Visualizations

### 5.1 Word Clouds for Positive and Negative Vocabularies

In [ ]:
from wordcloud import WordCloud

pos_text = ' '.join(df[df['label'] == 'positive']['clean_text'])
neg_text = ' '.join(df[df['label'] == 'negative']['clean_text'])

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

wc_pos = WordCloud(width=400, height=300, background_color='white', colormap='Greens').generate(pos_text)
axes[0].imshow(wc_pos, interpolation='bilinear')
axes[0].set_title('Positive Words', fontsize=14)
axes[0].axis('off')

wc_neg = WordCloud(width=400, height=300, background_color='white', colormap='Reds').generate(neg_text)
axes[1].imshow(wc_neg, interpolation='bilinear')
axes[1].set_title('Negative Words', fontsize=14)
axes[1].axis('off')

plt.tight_layout()
plt.show()

---
## 6. Model Training & Cross-Validation

Comparing Naive Bayes, Logistic Regression, SVM, and Random Forest with TF-IDF features.

In [ ]:
models = {
    'Naive Bayes': MultinomialNB(),
    'Logistic Regression': LogisticRegression(max_iter=1000, random_state=42),
    'Linear SVM': LinearSVC(max_iter=10000, random_state=42),
    'Random Forest': RandomForestClassifier(n_estimators=100, random_state=42)
}

cv_split = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
results = {}

for name, model in models.items():
    pipe = Pipeline([
        ('tfidf', TfidfVectorizer(ngram_range=(1, 2), max_features=500)),
        ('clf', model)
    ])
    scores = cross_val_score(pipe, df['clean_text'], y, cv=cv_split, scoring='accuracy')
    results[name] = scores
    print(f'{name:22s} -> Mean Accuracy: {scores.mean():.3f} (std: {scores.std():.3f})')

### 6.1 Model Comparison Bar Chart

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))
means = [results[m].mean() for m in models]
stds = [results[m].std() for m in models]

bars = ax.bar(models.keys(), means, yerr=stds, capsize=8, color=['#4C72B0', '#DD8452', '#55A868', '#C44E52'])
ax.set_ylabel('Cross-Validation Accuracy', fontsize=12)
ax.set_title('Model Comparison (5-Fold CV)', fontsize=14)
ax.set_ylim(0, 1.1)

for bar, mean in zip(bars, means):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02, f'{mean:.3f}',
            ha='center', fontsize=11, fontweight='bold')

plt.tight_layout()
plt.show()

---
## 7. Train / Test Split & Evaluation

Using Logistic Regression (best performer) for detailed analysis.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    df['clean_text'], y, test_size=0.3, random_state=42, stratify=y
)

final_pipe = Pipeline([
    ('tfidf', TfidfVectorizer(ngram_range=(1, 2), max_features=500)),
    ('clf', LogisticRegression(max_iter=1000, random_state=42))
])
final_pipe.fit(X_train, y_train)
y_pred = final_pipe.predict(X_test)
y_prob = final_pipe.predict_proba(X_test)[:, 1]

### 7.1 Confusion Matrix

In [ ]:
cm = confusion_matrix(y_test, y_pred)
fig, ax = plt.subplots(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=['Negative', 'Positive'],
            yticklabels=['Negative', 'Positive'], ax=ax)
ax.set_title('Confusion Matrix - Logistic Regression', fontsize=13)
ax.set_xlabel('Predicted')
ax.set_ylabel('Actual')
plt.tight_layout()
plt.show()

### 7.2 Classification Report

In [ ]:
print('Classification Report:\n')
print(classification_report(y_test, y_pred, target_names=['Negative', 'Positive']))

### 7.3 ROC Curves for All Models

In [ ]:
fig, ax = plt.subplots(figsize=(8, 6))

for name, model in models.items():
    pipe = Pipeline([
        ('tfidf', TfidfVectorizer(ngram_range=(1, 2), max_features=500)),
        ('clf', model)
    ])
    pipe.fit(X_train, y_train)
    if hasattr(pipe.named_steps['clf'], 'predict_proba'):
        prob = pipe.predict_proba(X_test)[:, 1]
    else:
        prob = pipe.decision_function(X_test)
        prob = (prob - prob.min()) / (prob.max() - prob.min())
    fpr, tpr, _ = roc_curve(y_test, prob)
    roc_auc = auc(fpr, tpr)
    ax.plot(fpr, tpr, lw=2, label=f'{name} (AUC = {roc_auc:.3f})')

ax.plot([0, 1], [0, 1], 'k--', lw=1, label='Random')
ax.set_xlabel('False Positive Rate', fontsize=12)
ax.set_ylabel('True Positive Rate', fontsize=12)
ax.set_title('ROC Curves', fontsize=14)
ax.legend(loc='lower right')
plt.tight_layout()
plt.show()

---
## 8. Most Informative Features

Extract top positive and negative coefficients from Logistic Regression.

In [ ]:
feature_names = final_pipe.named_steps['tfidf'].get_feature_names_out()
coefs = final_pipe.named_steps['clf'].coef_.flatten()

top_n = 15
top_positive_idx = np.argsort(coefs)[-top_n:]
top_negative_idx = np.argsort(coefs)[:top_n]

top_positive = [(feature_names[i], coefs[i]) for i in top_positive_idx][::-1]
top_negative = [(feature_names[i], coefs[i]) for i in top_negative_idx]

print('Top Positive Indicators:')
for word, val in top_positive:
    print(f'  {word:20s} {val:.3f}')

print('\nTop Negative Indicators:')
for word, val in top_negative:
    print(f'  {word:20s} {val:.3f}')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

pos_words = [w for w, v in top_positive]
pos_vals = [v for w, v in top_positive]
axes[0].barh(range(len(pos_words)), pos_vals, color='green')
axes[0].set_yticks(range(len(pos_words)))
axes[0].set_yticklabels(pos_words)
axes[0].set_title('Top Positive Features', fontsize=13)
axes[0].set_xlabel('Coefficient')

neg_words = [w for w, v in top_negative]
neg_vals = [v for w, v in top_negative]
axes[1].barh(range(len(neg_words)), neg_vals, color='red')
axes[1].set_yticks(range(len(neg_words)))
axes[1].set_yticklabels(neg_words)
axes[1].set_title('Top Negative Features', fontsize=13)
axes[1].set_xlabel('Coefficient')

plt.tight_layout()
plt.show()

---
## 9. Testing on Custom Sentences

In [ ]:
custom_sentences = [
    'I absolutely love this product, it is fantastic',
    'This is the worst thing I have ever bought',
    'Pretty good quality, I am satisfied',
    'Not bad, could be better though',
    'Terrible experience, do not recommend',
    'Amazing service and wonderful team',
    'It was okay, nothing special',
]

for s in custom_sentences:
    cleaned = preprocess(s)
    pred = final_pipe.predict([cleaned])[0]
    sentiment = 'Positive' if pred == 1 else 'Negative'
    print(f'{sentiment:9s} -> {s}')

---
## 10. Error Analysis

Examine which test examples were misclassified.

In [ ]:
X_test_array = X_test.reset_index(drop=True) if hasattr(X_test, 'reset_index') else pd.Series(X_test)
misclassified_idx = np.where(y_pred != y_test)[0]

print(f'Misclassified {len(misclassified_idx)} / {len(y_test)} test examples\n')

orig_texts = df.iloc[X_test.index]['text'].values if hasattr(X_test, 'index') else X_test_array

for idx in misclassified_idx[:10]:
    true_label = 'Positive' if y_test[idx] == 1 else 'Negative'
    pred_label = 'Positive' if y_pred[idx] == 1 else 'Negative'
    orig_idx = X_test.index[idx] if hasattr(X_test, 'index') else idx
    actual_text = df.iloc[orig_idx]['text'] if hasattr(X_test, 'index') else X_test_array[idx]
    print(f'Actual: {true_label:8s} | Predicted: {pred_label:8s} | Text: {actual_text}')

In [ ]:
print('\n===== Sentiment Analysis Complete =====')
print('Key Takeaways:')
print('  - TF-IDF with bigrams captures phrase-level sentiment well')
print('  - Logistic Regression and Linear SVM offer best accuracy-interpretability trade-off')
print('  - Stemming reduces vocabulary size but can lose nuanced meaning')
print('  - Word clouds and coefficient analysis reveal clear positive/negative lexicons')